# Assignment 2: Email Intent Extraction System

Building an LLM-based system that reads student emails and extracts structured information.

Student ID: 12349031  
Course: 700.390 Advanced Topics in Neurocomputing  



In [1]:
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

Running in Google Colab


## 1. Install Required Packages

In [2]:
%%capture
!pip install -q transformers torch accelerate sentencepiece
print("Packages installed")

## 2. Import Libraries

In [3]:
import os
import json
import re
import time
import warnings
from datetime import datetime

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
print("Libraries imported")

Libraries imported


## 3. Check GPU

In [4]:
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU - using CPU (slower)")

GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


## 4. Prompt Contract

The system prompt defines the role, task, constraints, and output format for the LLM.

In [5]:
SYSTEM_PROMPT = """You are an academic administration assistant.
Your task is to read a student email and extract information from it.

Extract exactly these fields:
- student_name: full name of the student, or null if not given
- student_id: student ID number as a string, or null if not given
- email_intent: one of "register", "deregister", or "unknown" if unclear
- course_or_exam: the course or exam mentioned, or null if not given
- confidence: "high" if intent is clear, "medium" if somewhat clear, "low" if ambiguous
- missing_information: list of field names that are missing, e.g. ["student_id"]

Rules:
- Use ONLY what is written in the email. Do not guess or invent anything.
- If intent is ambiguous or unclear, set email_intent to "unknown" and confidence to "low".
- Return ONLY valid JSON. No explanation, no markdown, no extra text.

Output format:
{"student_name": "...", "student_id": "...", "email_intent": "...", "course_or_exam": "...", "confidence": "...", "missing_information": []}"""

print("Prompt contract defined")
print(f"Prompt length: {len(SYSTEM_PROMPT)} characters")

Prompt contract defined
Prompt length: 987 characters


## 5. Test Emails

5 test cases covering different scenarios as required by the assignment.

In [6]:
TEST_EMAILS = [
    {
        "id": 1,
        "scenario": "Clear registration request",
        "email": """Dear Professor,
I would like to register for the Advanced Topics in Neurocomputing course.
My name is John Smith and my student ID is 2045123.
Best regards,
John"""
    },
    {
        "id": 2,
        "scenario": "Clear deregistration request",
        "email": """Dear Professor,
I need to deregister from the Advanced Topics in Neurocomputing exam scheduled for next month.
My name is Maria Garcia, student ID 2045456.
Thank you,
Maria"""
    },
    {
        "id": 3,
        "scenario": "Missing student ID",
        "email": """Dear Professor,
I would like to register for the Quantum Computing module.
My name is Ahmed Hassan.
Kind regards,
Ahmed"""
    },
    {
        "id": 4,
        "scenario": "Ambiguous intent",
        "email": """Dear Professor,
I am writing regarding the Advanced Topics in Neurocomputing course.
I registered last semester but I am not sure if I should stay in the course or withdraw.
My name is Lisa Chen, student ID 2045789.
Thanks,
Lisa"""
    },
    {
        "id": 5,
        "scenario": "Extra irrelevant information",
        "email": """Dear Professor,
I hope you are having a great summer. I went hiking last weekend and it was wonderful.
Anyway, I wanted to let you know that I would like to register for the AI Agents course.
The weather here has been really nice lately.
My name is Tom Brown, student ID 2045321.
Please let me know if you need anything else from me.
Best wishes,
Tom"""
    }
]

print(f"{len(TEST_EMAILS)} test emails defined:")
for e in TEST_EMAILS:
    print(f"  Email {e['id']}: {e['scenario']}")

5 test emails defined:
  Email 1: Clear registration request
  Email 2: Clear deregistration request
  Email 3: Missing student ID
  Email 4: Ambiguous intent
  Email 5: Extra irrelevant information


## 6. Load Model

Using Qwen2.5-1.5B-Instruct - a general instruction model good at following JSON output format.

In [7]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Tokenizer loaded")

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()
device = next(model.parameters()).device
print(f"Model loaded on {device}")

Loading Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on cuda:0


## 7. Helper Functions

In [8]:
def extract_intent(email_text: str, temperature: float = 0.1) -> dict:
    """Send email to model and get structured JSON response"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Email:\n{email_text}"}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(device)

    gen_kwargs = dict(
        max_new_tokens=256,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False  # greedy - more deterministic for structured output
    )

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    start = inputs.input_ids.shape[1]
    raw_output = tokenizer.decode(outputs[0][start:], skip_special_tokens=True).strip()
    return raw_output


def parse_json_output(raw: str) -> dict:
    """Try to parse JSON from model output, handle common formatting issues"""
    # Try direct parse
    try:
        return json.loads(raw), None
    except:
        pass

    # Try to find JSON object in the output
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group()), None
        except:
            pass

    return None, f"Could not parse JSON from: {raw[:100]}"


def validate_json(data: dict) -> list:
    """Check that all required fields are present"""
    required = ["student_name", "student_id", "email_intent", "course_or_exam", "confidence", "missing_information"]
    missing = [f for f in required if f not in data]
    issues = []
    if missing:
        issues.append(f"Missing fields: {missing}")
    if data.get("email_intent") not in ["register", "deregister", "unknown", None]:
        issues.append(f"Invalid email_intent: {data.get('email_intent')}")
    if data.get("confidence") not in ["high", "medium", "low", None]:
        issues.append(f"Invalid confidence: {data.get('confidence')}")
    return issues

print("Helper functions ready")

Helper functions ready


## 8. Run Extraction on All 5 Emails

In [9]:
results = []

for email_data in TEST_EMAILS:
    print(f"\n{'='*60}")
    print(f"Email {email_data['id']}: {email_data['scenario']}")
    print(f"{'='*60}")
    print(f"\nInput:\n{email_data['email']}\n")

    start = time.time()
    raw_output = extract_intent(email_data['email'])
    elapsed = time.time() - start

    parsed, parse_error = parse_json_output(raw_output)

    if parsed:
        validation_issues = validate_json(parsed)
        valid = len(validation_issues) == 0
    else:
        validation_issues = [parse_error]
        valid = False

    result = {
        "email_id": email_data['id'],
        "scenario": email_data['scenario'],
        "email_text": email_data['email'],
        "raw_output": raw_output,
        "parsed_json": parsed,
        "valid_json": valid,
        "validation_issues": validation_issues,
        "generation_time": elapsed,
        "timestamp": datetime.now().isoformat()
    }
    results.append(result)

    print(f"Raw output:\n{raw_output}")
    print(f"\nParsed JSON: {'OK' if valid else 'ISSUES: ' + str(validation_issues)}")
    print(f"Time: {elapsed:.1f}s")

print(f"\n\nAll {len(results)} emails processed.")


Email 1: Clear registration request

Input:
Dear Professor,
I would like to register for the Advanced Topics in Neurocomputing course.
My name is John Smith and my student ID is 2045123.
Best regards,
John

Raw output:
{"student_name": "John Smith", "student_id": "2045123", "email_intent": "register", "course_or_exam": "Advanced Topics in Neurocomputing", "confidence": "high", "missing_information": []}

Parsed JSON: OK
Time: 3.1s

Email 2: Clear deregistration request

Input:
Dear Professor,
I need to deregister from the Advanced Topics in Neurocomputing exam scheduled for next month.
My name is Maria Garcia, student ID 2045456.
Thank you,
Maria

Raw output:
{"student_name": "Maria Garcia", "student_id": "2045456", "email_intent": "deregister", "course_or_exam": "Advanced Topics in Neurocomputing", "confidence": "high", "missing_information": []}

Parsed JSON: OK
Time: 1.9s

Email 3: Missing student ID

Input:
Dear Professor,
I would like to register for the Quantum Computing module.

## 9. Save Results

In [10]:
os.makedirs('results', exist_ok=True)

# Save all results to JSON
with open('results/extraction_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved extraction_results.json")

# Save each email's output individually
for r in results:
    fname = f"results/email_{r['email_id']}_{r['scenario'].lower().replace(' ', '_')}.json"
    with open(fname, 'w') as f:
        json.dump({
            'scenario': r['scenario'],
            'email': r['email_text'],
            'output': r['parsed_json'],
            'valid': r['valid_json']
        }, f, indent=2)

print(f"Saved {len(results)} individual result files")

Saved extraction_results.json
Saved 5 individual result files


## 10. Results Summary

In [11]:
print("="*60)
print("RESULTS SUMMARY")
print("="*60)

valid_count = sum(1 for r in results if r['valid_json'])
print(f"Valid JSON output: {valid_count}/{len(results)}")
print(f"Average generation time: {sum(r['generation_time'] for r in results)/len(results):.1f}s")

print("\nPer email breakdown:")
for r in results:
    status = 'VALID' if r['valid_json'] else 'INVALID'
    j = r['parsed_json'] or {}
    print(f"\n  Email {r['email_id']} ({r['scenario']}): {status}")
    if j:
        print(f"    intent     : {j.get('email_intent')}")
        print(f"    confidence : {j.get('confidence')}")
        print(f"    missing    : {j.get('missing_information')}")

RESULTS SUMMARY
Valid JSON output: 5/5
Average generation time: 2.0s

Per email breakdown:

  Email 1 (Clear registration request): VALID
    intent     : register
    confidence : high
    missing    : []

  Email 2 (Clear deregistration request): VALID
    intent     : deregister
    confidence : high
    missing    : []

  Email 3 (Missing student ID): VALID
    intent     : register
    confidence : high
    missing    : []

  Email 4 (Ambiguous intent): VALID
    intent     : register
    confidence : high
    missing    : []

  Email 5 (Extra irrelevant information): VALID
    intent     : register
    confidence : high
    missing    : []


## 11. Explanation for Edge Cases

Notes on the missing and ambiguous cases.

In [12]:
EXPLANATIONS = {
    3: "Email 3 has no student ID. The model should return student_id as null and include 'student_id' in missing_information. Confidence should be high since the intent (register) is clear, but the missing ID means the request cannot be processed without follow-up.",
    4: "Email 4 is ambiguous - the student says they registered before but are unsure whether to stay or withdraw. The model should set email_intent to 'unknown' and confidence to 'low'. No action can be taken without a clearer request.",
    5: "Email 5 includes several sentences that are completely unrelated to the registration request (weather, hiking). The model should ignore these and extract only the relevant information. The intent and student details are still clearly stated."
}

print("Explanations for edge cases:\n")
for email_id, explanation in EXPLANATIONS.items():
    print(f"Email {email_id}:")
    print(f"  {explanation}")
    print()

Explanations for edge cases:

Email 3:
  Email 3 has no student ID. The model should return student_id as null and include 'student_id' in missing_information. Confidence should be high since the intent (register) is clear, but the missing ID means the request cannot be processed without follow-up.

Email 4:
  Email 4 is ambiguous - the student says they registered before but are unsure whether to stay or withdraw. The model should set email_intent to 'unknown' and confidence to 'low'. No action can be taken without a clearer request.

Email 5:
  Email 5 includes several sentences that are completely unrelated to the registration request (weather, hiking). The model should ignore these and extract only the relevant information. The intent and student details are still clearly stated.



## 12. Download Results

In [13]:
if IN_COLAB:
    import shutil
    shutil.make_archive('assignment_2_results', 'zip', '.')
    from google.colab import files
    files.download('assignment_2_results.zip')
    print("Downloaded!")
else:
    print("Results saved in results/ folder")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded!
